In [1]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd
from typing import Any, List
import numpy as np

# Merge Json of each image to a single json file

In [2]:
# EDIT THESE TWO PATHS
SRC_ROOT = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels")  # the folder containing train/ val/ test/
OUT_DIR = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val")

In [3]:
#Folder check 
for s in ["train", "val", "test"]:
    n = len(list((SRC_ROOT / s).glob("*.json"))) #glob finds all path names matching specific pattern and returns Generator object 
    print(s, n)

train 70000
val 10000
test 20000


In [4]:
#Peeking at one file

file = sorted((SRC_ROOT / "train").glob("*.json"))[0]
file_content = json.load(open(file))
print("File name :", file.name)
print("Json Keys :", list(file_content.keys()))
print("Json attributes :", file_content.get("attributes"))

File name : 0000f77c-6257be58.json
Json Keys : ['name', 'frames', 'attributes']
Json attributes : {'weather': 'clear', 'scene': 'city street', 'timeofday': 'daytime'}


In [5]:
# Testing

path = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels/train/0a0a0b1a-7c39d841.json")
with open(path) as fh:
    d = json.load(fh)
    items, weather, n_boxes, n_no_attr = [], Counter(), 0, 0

    name = d.get("name") or f.stem
    if not name.endswith(".jpg"):
        name += ".jpg"

        # per-image files store objects under frames[0]["objects"] (or "labels")
    objs = d["frames"][0].get("objects", []) 

    labels = [
        {"category": o["category"], "box2d": o["box2d"], "attributes": o.get("attributes", {})}
        for o in objs
        if o.get("box2d") is not None  # skip lanes / drivable area
        ]
    attrs = d.get("attributes") or {}
    n_no_attr += not attrs
    weather[attrs.get("weather", "missing")] += 1
    print(weather)
    n_boxes += len(labels)
    items.append({"name": name, "attributes": attrs, "labels": labels})
    print(items)


Counter({'clear': 1})
[{'name': '0a0a0b1a-7c39d841.jpg', 'attributes': {'weather': 'clear', 'scene': 'highway', 'timeofday': 'daytime'}, 'labels': [{'category': 'car', 'box2d': {'x1': 555.647397, 'y1': 304.228432, 'x2': 574.015906, 'y2': 316.474104}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 554.116689, 'y1': 318.004813, 'x2': 567.89307, 'y2': 328.719775}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 523.502507, 'y1': 311.881977, 'x2': 544.932435, 'y2': 327.189068}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 492.888326, 'y1': 311.881977, 'x2': 526.563926, 'y2': 337.904031}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'truck', 'box2d': {'x1': 393.392235, 'y1': 302.697723, 'x2': 453.089891, 'y2': 347.088286}, 

In [7]:
# To merge all jsons into one json

def convert_split(src_dir: Path, out_path: Path):
    files = sorted(src_dir.glob("*.json"))
    assert files, f"No .json files found in {src_dir}"

    items, weather, n_boxes, n_no_attr = [], Counter(), 0, 0
    for f in files:
        with open(f) as fh:
            d = json.load(fh)

        name = d.get("name") or f.stem
        if not name.endswith(".jpg"):
            name += ".jpg"

        # per-image files store objects under frames[0]["objects"] (or "labels")
        objs = d["frames"][0].get("objects", []) if d.get("frames") else d.get("labels", [])

        labels = [
            {"category": o["category"], "box2d": o["box2d"], "attributes": o.get("attributes", {})}
            for o in objs
            if o.get("box2d") is not None  # skip lanes / drivable area
        ]
        attrs = d.get("attributes") or {}
        n_no_attr += not attrs
        weather[attrs.get("weather", "missing")] += 1
        n_boxes += len(labels)
        items.append({"name": name, "attributes": attrs, "labels": labels})

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as fh:
        json.dump(items, fh)

    print(f"{src_dir.name}: {len(items)} images, {n_boxes} boxes -> {out_path}")
    print(f"  weather values: {dict(weather)}")
    if n_no_attr:
        print(f"  WARNING: {n_no_attr} files had no image-level attributes")


for split in SPLITS:
    convert_split(SRC_ROOT / split, OUT_DIR / f"bdd100k_labels_images_{split}.json")

NameError: name 'SPLITS' is not defined

In [8]:
path = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels/val")
convert_split(path,OUT_DIR)
SPLITS = ["val"]

val: 10000 images, 185578 boxes -> /Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val
  weather values: {'overcast': 1239, 'clear': 5346, 'undefined': 1157, 'rainy': 738, 'snowy': 769, 'partly cloudy': 738, 'foggy': 13}


# To capture count of various conditions - weather , scene , time of the day

In [9]:
def summarize_conditions(train: list[dict], val: list[dict]) -> pd.DataFrame:
    """Count images per weather / scene / time of day for each split."""
    rows = []
    for split, items in [("train", train), ("val", val)]:
        print("split",split)
        print("items",items)
        n = len(items) # no of images
        print('n:', n)
        for key in ["weather", "scene", "timeofday"]:
            counts = Counter(x["attributes"].get(key, "missing") for x in items)
            for value, c in counts.most_common():
                print(value , " : " ,c)
                rows.append(
                    {"split": split, "attribute": key, "value": value,
                     "images": c, "pct": round(100 * c / n, 1)}
                )
    return pd.DataFrame(rows)

In [10]:
#Test

val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"
train_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/train1.json"
train = json.load(open(train_path))
val = json.load(open(val_path))
summarize_conditions(train, val)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



,split,attribute,value,images,pct
0,train,weather,overcast,1,50.0
1,train,weather,clear,1,50.0
2,train,scene,city street,1,50.0
3,train,scene,highway,1,50.0
4,train,timeofday,daytime,1,50.0
5,train,timeofday,night,1,50.0
6,val,weather,clear,5346,53.5
7,val,weather,overcast,1239,12.4
8,val,weather,undefined,1157,11.6
9,val,weather,snowy,769,7.7


In [11]:
#Test

val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"

val = json.load(open(val_path))

print(type(val), len(val))   # <class 'list'> 10000
print(val[0]["attributes"])

rows = []
n = len(val)
print(val)
for key in ["weather", "scene", "timeofday"]:
    counts = Counter(x["attributes"].get(key, "missing") for x in val)
    for value, c in counts.most_common():
        print(value)
        print("c", c)
        rows.append({"attribute": key, "value": value, "images": c, "pct": round(100 * c / n, 1)})

pd.DataFrame(rows)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



,attribute,value,images,pct
0,weather,clear,5346,53.5
1,weather,overcast,1239,12.4
2,weather,undefined,1157,11.6
3,weather,snowy,769,7.7
4,weather,rainy,738,7.4
5,weather,partly cloudy,738,7.4
6,weather,foggy,13,0.1
7,scene,city street,6112,61.1
8,scene,highway,2499,25.0
9,scene,residential,1253,12.5


In [ ]:
# Records to tables

In [12]:
def items_to_tables(items: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Turn image records into an images table and a boxes table."""
    imgs, boxes = [], []
    for item in items:
        print('item:' , item)
        name = item["name"]
        attrs = item.get("attributes") or {}
        print('Path:' , Path(name), Path(name).stem, Path(name).stem.split("-"))
        imgs.append(
            dict(
                name=name,
                video_id=Path(name).stem.split("-")[0],  # prefix = source video
                weather=attrs.get("weather", "undefined"),
                scene=attrs.get("scene", "undefined"),
                timeofday=attrs.get("timeofday", "undefined"),
            )
        )
        print("imgs:",imgs)
        for lab in item.get("labels") or []:
            print("lab:", lab)
            b = lab.get("box2d")
            print("box:", b)
            if b is None:
                continue
            la = lab.get("attributes") or {}
            print("trafficLightColor:", type(la.get("trafficLightColor", "none")))
            print("label attributes:",lab)
            boxes.append(
                dict(
                    name=name, category=lab["category"],
                    x1=b["x1"], y1=b["y1"], x2=b["x2"], y2=b["y2"],
                    occluded=bool(la.get("occluded", False)),
                    truncated=bool(la.get("truncated", False)),
                    traffic_light_color=la.get("trafficLightColor", "none")
                )
            )
    #box_cols = ["name", "category", "x1", "y1", "x2", "y2", "occluded", "truncated", "traffic_light_color"]
    return pd.DataFrame(imgs), pd.DataFrame(boxes)#, columns=box_cols)

In [13]:
val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"
val = json.load(open(val_path))
df1, df2 = items_to_tables(val)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [14]:
df1

,name,video_id,weather,scene,timeofday
0,b1c66a42-6f7d68ca.jpg,b1c66a42,overcast,city street,daytime
1,b1c81faa-3df17267.jpg,b1c81faa,clear,highway,night
2,b1c81faa-c80764c5.jpg,b1c81faa,clear,highway,night
3,b1c9c847-3bda4659.jpg,b1c9c847,undefined,city street,daytime
4,b1ca2e5d-84cf9134.jpg,b1ca2e5d,clear,city street,night
...,...,...,...,...,...
9995,fe1d9184-d144106a.jpg,fe1d9184,clear,city street,daytime
9996,fe1d9184-dec09b65.jpg,fe1d9184,overcast,city street,daytime
9997,fe1f2409-5b415eb7.jpg,fe1f2409,clear,residential,daytime
9998,fe1f2409-c16ea1ed.jpg,fe1f2409,clear,city street,daytime


In [59]:
df2

,name,category,x1,y1,x2,y2,occluded,truncated,traffic_light_color
0,b1c66a42-6f7d68ca.jpg,traffic sign,1000.698742,281.992415,1040.626872,326.911560,False,False,none
1,b1c66a42-6f7d68ca.jpg,traffic sign,214.613695,172.190058,274.505889,229.586743,False,False,none
2,b1c66a42-6f7d68ca.jpg,traffic sign,797.314833,313.186265,829.756437,341.884608,False,False,none
3,b1c66a42-6f7d68ca.jpg,traffic sign,652.575363,303.204232,685.016968,315.681772,False,False,none
4,b1c66a42-6f7d68ca.jpg,traffic light,707.476543,311.938510,716.210821,328.159313,False,False,green
...,...,...,...,...,...,...,...,...,...
185573,fe1f55fa-19ba3600.jpg,person,491.615094,388.051507,511.726280,444.314645,False,False,none
185574,fe1f55fa-19ba3600.jpg,person,477.889798,361.848672,496.606109,408.015571,True,False,none
185575,fe1f55fa-19ba3600.jpg,person,583.948891,384.308246,611.399480,431.722899,True,False,none
185576,fe1f55fa-19ba3600.jpg,person,405.520064,408.015571,415.502096,429.227390,False,False,none


In [11]:
#Clean boxes

In [15]:
df3=df2[["x1", "x2"]].copy()
df3[["x1", "x2"]] = [-10,1493.56]
df3

,x1,x2
0,-10,1493.56
1,-10,1493.56
2,-10,1493.56
3,-10,1493.56
4,-10,1493.56
...,...,...
185573,-10,1493.56
185574,-10,1493.56
185575,-10,1493.56
185576,-10,1493.56


In [13]:
df3[["x1", "x2"]].clip(0, 1280)

,x1,x2
0,0,1280.0


In [15]:
#min_box_px =2: Very small bounding boxes can represent tiny objects or annotation noise. 
#Filtering them helps avoid training on boxes that may be difficult to detect reliably.

def _build_pool(images:pd.DataFrame, boxes:pd.DataFrame, images_dir: Path, classes: List, min_box_px: float, IMG_W:int, IMG_H:int):
    b = boxes[boxes.category.isin(classes)].copy()
    print(b)
    b[["x1", "x2"]] = b[["x1", "x2"]].clip(0, IMG_W)
    b[["y1", "y2"]] = b[["y1", "y2"]].clip(0, IMG_H)
    b["w"] = b.x2 - b.x1
    b["h"] = b.y2 - b.y1
    b["area"] = b.w * b.h
    b = b[(b.w >= min_box_px) & (b.h >= min_box_px)].reset_index(drop=True) #discard bounding box whose width and height less than 2 pixels
    print('b:', b)
    print('images:',images[images.name.isin(b.name)])
    imgs = images[images.name.isin(b.name)].copy()
    exists = imgs.name.map(lambda n: (images_dir / n).exists())
    print('exists:',list(exists))
    n_missing = int((~exists).sum())
    if n_missing:
        print(f"labelled images not found in {n_missing} skipping them.", images_dir)
    imgs = imgs[exists].reset_index(drop=True)
    if imgs.empty:
        raise FileNotFoundError(
            f"No labelled images found in {images_dir}. Check data_prep.images_root in parameters."
        )
    b = b[b.name.isin(imgs.name)].reset_index(drop=True)
    imgs["stratum"] = imgs.timeofday + "|" + imgs.weather
    print(f"Pool from {images_dir}: {len(imgs)}, {len(b)} boxes")
    return imgs, b

In [16]:
images_dir = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/imgs")
print(images_dir)
imgs,b = _build_pool(df1, df2, images_dir, ['car', 'pedestrian', 'truck', 'traffic light', 'traffic sign', 'rider'], 2.0, 1280, 720)

/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/imgs
                         name       category           x1          y1  \
0       b1c66a42-6f7d68ca.jpg   traffic sign  1000.698742  281.992415   
1       b1c66a42-6f7d68ca.jpg   traffic sign   214.613695  172.190058   
2       b1c66a42-6f7d68ca.jpg   traffic sign   797.314833  313.186265   
3       b1c66a42-6f7d68ca.jpg   traffic sign   652.575363  303.204232   
4       b1c66a42-6f7d68ca.jpg  traffic light   707.476543  311.938510   
...                       ...            ...          ...         ...   
185569  fe1f55fa-19ba3600.jpg   traffic sign   113.545617  334.398083   
185570  fe1f55fa-19ba3600.jpg          truck   426.731884  376.821721   
185571  fe1f55fa-19ba3600.jpg          truck   447.943700  336.893592   
185572  fe1f55fa-19ba3600.jpg   traffic sign   606.408465  310.690757   
185577  fe1f55fa-19ba3600.jpg   traffic sign   366.839689  386.803753   

                 x2  

In [17]:
# When running node 3 : build_test_pool it returned 
#Pool from data/01_raw/bdd100k/images/val: 9999 images, 170741 boxes  
#To find out that 1 image below code

def items_to_tables(items: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Turn image json records into an images table and a boxes table."""
    
    imgs, boxes = [], []
    for item in items:
        name = item["name"]
        attrs = item.get("attributes") or {}
        imgs.append(
            dict(
                name=name,
                video_id=Path(name).stem.split("-")[0],  # prefix = source video
                weather=attrs.get("weather", "undefined"),
                scene=attrs.get("scene", "undefined"),
                timeofday=attrs.get("timeofday", "undefined"),
            )
        )
        for label in item.get("labels") or []:
            box_coords = label.get("box2d")
            if box_coords is None:
                continue
            label_attrs = label.get("attributes") or {}
            boxes.append(
                dict(
                    name=name, category=label["category"],
                    x1=box_coords["x1"], y1=box_coords["y1"], x2=box_coords["x2"], y2=box_coords["y2"],
                    occluded=bool(label_attrs.get("occluded", False)),
                    truncated=bool(label_attrs.get("truncated", False)),
                    traffic_light_color=label_attrs.get("trafficLightColor", "none"),
                )
            )
    #box_cols = ["name", "category", "x1", "y1", "x2", "y2", "occluded", "truncated"]
    return pd.DataFrame(imgs), pd.DataFrame(boxes), #columns=box_cols)


import json
from pathlib import Path

# 1. Load the merged val labels (Kedro's output; use your own val.json path if different)
val = json.load(open("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/02_intermediate/bdd_labels_val.json"))

# 2. Build the two tables (needs your items_to_tables cell to have been run)
val_images_raw, val_boxes_raw = items_to_tables(val)
print(len(val_images_raw), len(val_boxes_raw))

# 3. Build the pool (needs IMG_W, IMG_H, logger and _build_pool to have been run)
classes = ["car", "pedestrian", "truck", "bus", "motorcycle", "bicycle", "rider", "traffic light", "traffic sign"]
images_dir = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/images/val")
pool_imgs, pool_boxes = _build_pool(val_images_raw, val_boxes_raw, images_dir, classes, 2.0, 1280, 720)

# 4. Which image was dropped, and why?
missing = set(val_images_raw.name) - set(pool_imgs.name)
print("dropped:", missing)

m = next(iter(missing))
print(val_boxes_raw[val_boxes_raw.name == m].category.value_counts())
print("file exists:", (images_dir / m).exists())
print("To check classes:" , val_boxes_raw.category.value_counts())

10000 185578
                         name       category           x1          y1  \
0       b1c66a42-6f7d68ca.jpg   traffic sign  1000.698742  281.992415   
1       b1c66a42-6f7d68ca.jpg   traffic sign   214.613695  172.190058   
2       b1c66a42-6f7d68ca.jpg   traffic sign   797.314833  313.186265   
3       b1c66a42-6f7d68ca.jpg   traffic sign   652.575363  303.204232   
4       b1c66a42-6f7d68ca.jpg  traffic light   707.476543  311.938510   
...                       ...            ...          ...         ...   
185569  fe1f55fa-19ba3600.jpg   traffic sign   113.545617  334.398083   
185570  fe1f55fa-19ba3600.jpg          truck   426.731884  376.821721   
185571  fe1f55fa-19ba3600.jpg          truck   447.943700  336.893592   
185572  fe1f55fa-19ba3600.jpg   traffic sign   606.408465  310.690757   
185577  fe1f55fa-19ba3600.jpg   traffic sign   366.839689  386.803753   

                 x2          y2  occluded  truncated traffic_light_color  
0       1040.626872  326.911560    

In [1]:
# 4. Stratified sampling and leakage-aware split 


In [18]:
def _stratified_sample(df, n, seed):
    """Sample ~n rows keeping the (timeofday|weather) mix; rare strata keep >=1."""
    rng = np.random.default_rng(seed)
    n = min(n, len(df))
    frac = n / len(df)
    parts = []
    for _, g in df.groupby("stratum"):
        k = min(len(g), max(1, int(round(len(g) * frac)))) #when rounded grouped datapoints can become 0 so at max 1 need to be taken
        parts.append(g.sample(n=k, random_state=int(rng.integers(1_000_000_000))))
    out = pd.concat(parts)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)
    return out.reset_index(drop=True)

In [19]:
rng = np.random.default_rng(42)
random_state=int(rng.integers(1_000_000_000))
random_state

89250953

In [20]:
rng = np.random.default_rng(42)
n=min(2000, len(pool_imgs))
frac = n/len(pool_imgs)
frac
parts=[]
for _, g in pool_imgs.groupby("stratum"):
    #print(_,":",min(len(g),max(1, int(round(len(g)*frac)))))
    k=min(len(g),max(1, int(round(len(g)*frac))))
    parts.append(g.sample(n=k,random_state=int(rng.integers(1_000_000_000))))
out = pd.concat(parts)   
print(out)
if len(out) > n:
    out = out.sample(n=n, random_state=42)
out.reset_index(drop=True)

                       name  video_id    weather        scene  timeofday  \
2116  b6ffe7bd-f7d2cdce.jpg  b6ffe7bd      clear  city street  dawn/dusk   
5775  c0677163-cccbd1cf.jpg  c0677163      clear      highway  dawn/dusk   
4479  bd3bd3ab-cfd5b40e.jpg  bd3bd3ab      clear      highway  dawn/dusk   
297   b26dddb5-28e4a05f.jpg  b26dddb5      clear  city street  dawn/dusk   
4074  bc2e52cc-cbd2ddae.jpg  bc2e52cc      clear  city street  dawn/dusk   
...                     ...       ...        ...          ...        ...   
1479  b568371d-a596d2d4.jpg  b568371d  undefined       tunnel  undefined   
7016  c33ff185-93a0dd91.jpg  c33ff185  undefined       tunnel  undefined   
7034  c356bbec-10171932.jpg  c356bbec  undefined       tunnel  undefined   
8880  c7b32a0d-ffed4ed6.jpg  c7b32a0d  undefined  parking lot  undefined   
8171  c602b3c5-0bba271b.jpg  c602b3c5  undefined       tunnel  undefined   

                  stratum  
2116      dawn/dusk|clear  
5775      dawn/dusk|clear  
447

,name,video_id,weather,scene,timeofday,stratum
0,b4c86653-a5abaf4e.jpg,b4c86653,clear,city street,night,night|clear
1,bd1d310c-f5fb429f.jpg,bd1d310c,clear,city street,daytime,daytime|clear
2,b58f8128-3d42f144.jpg,b58f8128,clear,city street,night,night|clear
3,b4e927af-828f4a9b.jpg,b4e927af,rainy,highway,daytime,daytime|rainy
4,bbcd82d7-522d7b83.jpg,bbcd82d7,clear,city street,night,night|clear
...,...,...,...,...,...,...
1995,bb185003-ebc11af9.jpg,bb185003,clear,city street,night,night|clear
1996,b37cacf7-0d3ce003.jpg,b37cacf7,undefined,city street,daytime,daytime|undefined
1997,c3c0f47b-1e1b1003.jpg,c3c0f47b,undefined,city street,daytime,daytime|undefined
1998,ca40ddd3-f81f28e9.jpg,ca40ddd3,clear,city street,night,night|clear


In [22]:
pool_imgs[pool_imgs['stratum']=='dawn/dusk|clear']

,name,video_id,weather,scene,timeofday,stratum
10,b1ceb32e-51852abe.jpg,b1ceb32e,clear,city street,dawn/dusk,dawn/dusk|clear
23,b1d10d08-5b108225.jpg,b1d10d08,clear,city street,dawn/dusk,dawn/dusk|clear
26,b1d10d08-da110fcb.jpg,b1d10d08,clear,city street,dawn/dusk,dawn/dusk|clear
32,b1d4b62c-60aab822.jpg,b1d4b62c,clear,residential,dawn/dusk,dawn/dusk|clear
35,b1d59b1f-a38aec79.jpg,b1d59b1f,clear,city street,dawn/dusk,dawn/dusk|clear
...,...,...,...,...,...,...
9898,ca2c7c84-5495c318.jpg,ca2c7c84,clear,highway,dawn/dusk,dawn/dusk|clear
9909,ca309fd3-4a6f706c.jpg,ca309fd3,clear,city street,dawn/dusk,dawn/dusk|clear
9946,ca479701-2ab57f9a.jpg,ca479701,clear,city street,dawn/dusk,dawn/dusk|clear
9979,ca679a1e-cee7f0bc.jpg,ca679a1e,clear,city street,dawn/dusk,dawn/dusk|clear


In [23]:
pool_imgs

,name,video_id,weather,scene,timeofday,stratum
0,b1c66a42-6f7d68ca.jpg,b1c66a42,overcast,city street,daytime,daytime|overcast
1,b1c81faa-3df17267.jpg,b1c81faa,clear,highway,night,night|clear
2,b1c81faa-c80764c5.jpg,b1c81faa,clear,highway,night,night|clear
3,b1c9c847-3bda4659.jpg,b1c9c847,undefined,city street,daytime,daytime|undefined
4,b1ca2e5d-84cf9134.jpg,b1ca2e5d,clear,city street,night,night|clear
...,...,...,...,...,...,...
9994,fe1d9184-d144106a.jpg,fe1d9184,clear,city street,daytime,daytime|clear
9995,fe1d9184-dec09b65.jpg,fe1d9184,overcast,city street,daytime,daytime|overcast
9996,fe1f2409-5b415eb7.jpg,fe1f2409,clear,residential,daytime,daytime|clear
9997,fe1f2409-c16ea1ed.jpg,fe1f2409,clear,city street,daytime,daytime|clear


In [22]:
pool_boxes

,name,category,x1,y1,x2,y2,occluded,truncated,traffic_light_color,w,h,area
0,b1c66a42-6f7d68ca.jpg,traffic sign,1000.698742,281.992415,1040.626872,326.911560,False,False,none,39.928130,44.919145,1793.537461
1,b1c66a42-6f7d68ca.jpg,traffic sign,214.613695,172.190058,274.505889,229.586743,False,False,none,59.892194,57.396685,3437.613393
2,b1c66a42-6f7d68ca.jpg,traffic sign,797.314833,313.186265,829.756437,341.884608,False,False,none,32.441604,28.698343,931.020279
3,b1c66a42-6f7d68ca.jpg,traffic sign,652.575363,303.204232,685.016968,315.681772,False,False,none,32.441605,12.477540,404.791424
4,b1c66a42-6f7d68ca.jpg,traffic light,707.476543,311.938510,716.210821,328.159313,False,False,green,8.734278,16.220803,141.677003
...,...,...,...,...,...,...,...,...,...,...,...,...
170736,fe1f55fa-19ba3600.jpg,traffic sign,113.545617,334.398083,136.005191,350.618886,False,False,none,22.459574,16.220803,364.312325
170737,fe1f55fa-19ba3600.jpg,truck,426.731884,376.821721,449.191454,427.979637,True,False,none,22.459570,51.157916,1148.984795
170738,fe1f55fa-19ba3600.jpg,truck,447.943700,336.893592,529.047713,444.200439,True,False,none,81.104013,107.306847,8703.015914
170739,fe1f55fa-19ba3600.jpg,traffic sign,606.408465,310.690757,631.363546,353.114394,False,False,none,24.955081,42.423637,1058.685298


In [21]:
def _group_split(df, val_frac, seed):
    """Split by video id within each stratum so no video spans train and val."""
    rng = np.random.default_rng(seed)
    val_videos = set()
    for _, g in df.groupby("stratum"):
        vids = g.video_id.unique()
        rng.shuffle(vids)
        k = int(round(len(vids) * val_frac))
        val_videos.update(vids[:k])
    is_val = df.video_id.isin(val_videos)
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)

In [25]:
rng = np.random.default_rng(42)
val_frac = 0.20
val_videos = set()
for _, g in pool_imgs.groupby("stratum"):
    vids = g.video_id.unique()
    rng.shuffle(vids)
    #print(vids)
    k=int(round(len(vids) * val_frac))
    s = set(vids[:k])
    #print("already present:",len(s),s.intersection(val_videos))
    val_videos.update(vids[:k])

is_val = pool_imgs.video_id.isin(val_videos)
print(~is_val)  
    

0        True
1        True
2        True
3        True
4        True
        ...  
9994    False
9995    False
9996     True
9997     True
9998     True
Name: video_id, Length: 9999, dtype: bool


/var/folders/nj/q6tk_8_n12g79rlp2jb0tg080000gn/T/ipykernel_2503/1163953854.py:6: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(vids)


In [25]:
pool_imgs.groupby("video_id")["stratum"].nunique().value_counts()

stratum
1    4821
2     844
3     184
4      64
5      13
6       4
7       1
Name: count, dtype: int64

In [22]:
def _check_no_leakage(splits: dict):
    print("input: ",splits)
    names = list(splits)
    print("names: ", names)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            overlap = set(splits[names[i]].video_id) & set(splits[names[j]].video_id)
            assert not overlap, f"Video leakage between {names[i]} and {names[j]}: {len(overlap)} videos"
    print("Leakage check passed: no video id appears in more than one split.")

In [23]:
def split_dataset(train_imgs, train_boxes, test_imgs, test_boxes, params: dict):
    """Build train / val / test manifests (images and boxes, with a `split` column)."""
    n_train, n_val, n_test, seed = (params[k] for k in ("n_train", "n_val", "n_test", "seed"))
 
    sampled = _stratified_sample(train_imgs, n_train + n_val, seed)
    tr, va = _group_split(sampled, n_val / (n_train + n_val), seed)
    te = _stratified_sample(test_imgs, n_test, seed + 1)
 
    # (images, boxes pool, official image folder the files live in)
    splits = {
        "train": (tr, train_boxes, "train"),
        "val": (va, train_boxes, "train"),
        "test": (te, test_boxes, "val"),
    }
    #_check_no_leakage({k: v[0] for k, v in splits.items()})
 
    img_parts, box_parts = [], []
    for split, (imgs, pool_boxes, src_dir) in splits.items():
        b = pool_boxes[pool_boxes.name.isin(imgs.name)]
        counts = b.groupby("name").size()
        img_parts.append(imgs.assign(
            split=split, source_dir=src_dir,
            n_boxes=imgs.name.map(counts).fillna(0).astype(int)))
        box_parts.append(b.assign(split=split))
        #logger.info("%s: %d images, %d boxes", split, len(imgs), len(b))
 
    return pd.concat(img_parts, ignore_index=True), pd.concat(box_parts, ignore_index=True)

In [24]:
train_imgs=imgs
test_imgs = imgs
train_boxes=b
test_boxes =b
params = {"n_train": 40,
  "n_val": 15,
  "n_test": 30,
  "seed": 42}
split_img, split_box = split_dataset(train_imgs, train_boxes, test_imgs, test_boxes, params)

/var/folders/nj/q6tk_8_n12g79rlp2jb0tg080000gn/T/ipykernel_2715/3603774458.py:7: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(vids)


In [26]:
split_box

,name,category,x1,y1,x2,y2,occluded,truncated,traffic_light_color,w,h,area,split
0,b1ca2e5d-84cf9134.jpg,car,424.791155,285.540452,444.378067,308.799910,True,False,none,19.586912,23.259458,455.580957,train
1,b1ca2e5d-84cf9134.jpg,car,340.322596,285.540452,384.393147,323.490094,True,False,none,44.070551,37.949642,1672.461633,train
2,b1ca2e5d-84cf9134.jpg,car,372.151328,291.661362,434.584610,335.731914,True,False,none,62.433282,44.070552,2751.469201,train
3,b1ca2e5d-84cf9134.jpg,car,181.178935,259.832630,364.806235,387.147558,True,False,none,183.627300,127.314928,23378.496478,train
4,b1ca2e5d-84cf9134.jpg,car,146.901840,297.782272,182.403118,332.059368,True,False,none,35.501278,34.277096,1216.880714,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1286,b1f20aa0-6ef1db42.jpg,car,764.099298,281.176228,848.913742,388.877112,True,False,none,84.814444,107.700884,9134.590595,test
1287,b1f20aa0-6ef1db42.jpg,car,817.949739,277.137445,968.730974,417.148592,True,False,none,150.781235,140.011147,21111.053658,test
1288,b1f20aa0-6ef1db42.jpg,car,589.854668,298.869948,632.935020,337.911517,False,False,none,43.080352,39.041569,1681.924535,test
1289,b1f20aa0-6ef1db42.jpg,car,2.877630,173.667673,379.830718,535.811890,False,True,none,376.953088,362.144217,136511.380899,test


In [85]:
#check no leakage
splits = {"n_train": [1,2,3,4,5,6,7], "n_val":[2,8,9,4,5,6]}
names = list(params)
overlap = set(splits[names[0]]) & set(splits[names[1]])
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        overlap = set(splits[names[0]]) & set(splits[names[1]])
        assert not overlap, f"Video leakage between {names[i]} and {names[j]}: {len(overlap)} videos"


AssertionError: Video leakage between n_train and n_val: 4 videos

In [66]:
imgs

,name,video_id,weather,scene,timeofday,stratum
0,b1c66a42-6f7d68ca.jpg,b1c66a42,overcast,city street,daytime,daytime|overcast
1,b1c81faa-3df17267.jpg,b1c81faa,clear,highway,night,night|clear
2,b1c81faa-c80764c5.jpg,b1c81faa,clear,highway,night,night|clear
3,b1c9c847-3bda4659.jpg,b1c9c847,undefined,city street,daytime,daytime|undefined
4,b1ca2e5d-84cf9134.jpg,b1ca2e5d,clear,city street,night,night|clear
...,...,...,...,...,...,...
80,b1f20aa0-50213047.jpg,b1f20aa0,clear,city street,night,night|clear
81,b1f20aa0-6ef1db42.jpg,b1f20aa0,clear,city street,night,night|clear
82,b1f6c103-5ce1f3c6.jpg,b1f6c103,clear,highway,night,night|clear
83,b1f6c103-8b75ea3e.jpg,b1f6c103,clear,city street,night,night|clear


In [45]:
# 5. Write the YOLO dataset

In [50]:
def write_yolo_dataset(images_manifest, boxes_manifest, params: dict) -> str:
    """Copy (or symlink) images and write YOLO labels + dataset.yaml. Returns the yaml path."""
    out = Path(params["yolo_out_dir"])
    classes = params["classes"]
    class_to_id = {c: i for i, c in enumerate(classes)}
    images_root = Path(params["images_root"])
 
    # Start clean so reruns with a different sample never leave stale files behind.
    for sub in ("images", "labels"):
        shutil.rmtree(out / sub, ignore_errors=True)
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
 
    by_img = {k: g for k, g in boxes_manifest.groupby(["split", "name"])}
    for row in images_manifest.itertuples():
        src = (images_root / row.source_dir / row.name).resolve()
        dst = out / "images" / row.split / row.name
        if params["link_images"]:
            dst.symlink_to(src)
        else:
            shutil.copy2(src, dst)
 
        g = by_img[(row.split, row.name)]
        cx = ((g.x1 + g.x2) / 2 / IMG_W).round(6)
        cy = ((g.y1 + g.y2) / 2 / IMG_H).round(6)
        w = (g.w / IMG_W).round(6)
        h = (g.h / IMG_H).round(6)
        cls = g.category.map(class_to_id)
        lines = [f"{c} {x} {y} {ww} {hh}" for c, x, y, ww, hh in zip(cls, cx, cy, w, h)]
        (out / "labels" / row.split / f"{Path(row.name).stem}.txt").write_text("\n".join(lines) + "\n")
 
    yaml_lines = [
        f"path: {out.resolve()}", "train: images/train", "val: images/val", "test: images/test", "names:",
    ] + [f"  {i}: {c}" for i, c in enumerate(classes)]
    yaml_path = out / "dataset.yaml"
    yaml_path.write_text("\n".join(yaml_lines) + "\n")
    logger.info("YOLO dataset written to %s", out.resolve())
    return str(yaml_path)

In [53]:
classes = ["car", "person", "truck", "bus", "motor", "bike", "rider", "traffic light", "traffic sign"]
class_to_id = {c: i for i, c in enumerate(classes)}
class_to_id
out=Path("data/01_raw/bdd100k/images/label")
yaml_lines = [
        f"path: {out.resolve()}", "train: images/train", "val: images/val", "test: images/test", "names:",
    ] + [f"  {i}: {c}" for i, c in enumerate(classes)]
print(yaml_lines)

['path: /Users/shravanthij/Documents/ML academy/driving-perception-validation/notebooks/data/01_raw/bdd100k/images/label', 'train: images/train', 'val: images/val', 'test: images/test', 'names:', '  0: car', '  1: person', '  2: truck', '  3: bus', '  4: motor', '  5: bike', '  6: rider', '  7: traffic light', '  8: traffic sign']


In [ ]:
images_dir: data/01_raw/bdd100k/images/label

In [38]:
by_img = {k: g for k, g in split_box.groupby(["split", "name"])}


In [49]:
for row in split_img.itertuples():
    print(Path(row.name).stem)
    print(by_img[(row.split, row.name)])
    g = by_img[(row.split, row.name)]
    cx = ((g.x1 + g.x2) / 2 / IMG_W).round(6)
    cy = ((g.y1 + g.y2) / 2 / IMG_H).round(6)
    w = (g.w / IMG_W).round(6)
    h = (g.h / IMG_H).round(6)
    cls = g.category.map(class_to_id)

b1d7b3ac-9e14f05f
                      name      category           x1          y1  \
229  b1d7b3ac-9e14f05f.jpg  traffic sign   794.807529  383.064455   
230  b1d7b3ac-9e14f05f.jpg           car  1127.343004  375.382680   
231  b1d7b3ac-9e14f05f.jpg           car   666.521908  382.552337   
232  b1d7b3ac-9e14f05f.jpg           car   484.207810  385.625044   
233  b1d7b3ac-9e14f05f.jpg           car  1110.955220  371.456461   
234  b1d7b3ac-9e14f05f.jpg           car  1056.670686  368.383751   
235  b1d7b3ac-9e14f05f.jpg           car  1005.390592  373.231776   
236  b1d7b3ac-9e14f05f.jpg           car   930.621329  379.377197   
237  b1d7b3ac-9e14f05f.jpg           car   852.779352  376.304488   

              x2          y2  occluded  truncated traffic_light_color  \
229   811.195313  399.452241     False      False                none   
230  1162.252379  408.328975      True      False                none   
231   763.824374  433.764161     False      False                none   

NameError: name 'IMG_W' is not defined

In [40]:
split_box

,name,category,x1,y1,x2,y2,occluded,truncated,traffic_light_color,w,h,area,split
0,b1ca2e5d-84cf9134.jpg,car,424.791155,285.540452,444.378067,308.799910,True,False,none,19.586912,23.259458,455.580957,train
1,b1ca2e5d-84cf9134.jpg,car,340.322596,285.540452,384.393147,323.490094,True,False,none,44.070551,37.949642,1672.461633,train
2,b1ca2e5d-84cf9134.jpg,car,372.151328,291.661362,434.584610,335.731914,True,False,none,62.433282,44.070552,2751.469201,train
3,b1ca2e5d-84cf9134.jpg,car,181.178935,259.832630,364.806235,387.147558,True,False,none,183.627300,127.314928,23378.496478,train
4,b1ca2e5d-84cf9134.jpg,car,146.901840,297.782272,182.403118,332.059368,True,False,none,35.501278,34.277096,1216.880714,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1286,b1f20aa0-6ef1db42.jpg,car,764.099298,281.176228,848.913742,388.877112,True,False,none,84.814444,107.700884,9134.590595,test
1287,b1f20aa0-6ef1db42.jpg,car,817.949739,277.137445,968.730974,417.148592,True,False,none,150.781235,140.011147,21111.053658,test
1288,b1f20aa0-6ef1db42.jpg,car,589.854668,298.869948,632.935020,337.911517,False,False,none,43.080352,39.041569,1681.924535,test
1289,b1f20aa0-6ef1db42.jpg,car,2.877630,173.667673,379.830718,535.811890,False,True,none,376.953088,362.144217,136511.380899,test


In [45]:
split_img

,name,video_id,weather,scene,timeofday,stratum,split,source_dir,n_boxes
0,b1d7b3ac-9e14f05f.jpg,b1d7b3ac,clear,residential,dawn/dusk,dawn/dusk|clear,train,train,9
1,b1d7b3ac-afa57f22.jpg,b1d7b3ac,clear,residential,dawn/dusk,dawn/dusk|clear,train,train,3
2,b1d9e136-6c94ea3f.jpg,b1d9e136,undefined,city street,daytime,daytime|undefined,train,train,15
3,b1d7b3ac-36f2d3b7.jpg,b1d7b3ac,clear,residential,daytime,daytime|clear,train,train,6
4,b1d968b9-563405f4.jpg,b1d968b9,overcast,city street,daytime,daytime|overcast,train,train,15
...,...,...,...,...,...,...,...,...,...
80,b1d10d08-743fd86c.jpg,b1d10d08,clear,city street,night,night|clear,test,val,14
81,b1d0a191-de8948f6.jpg,b1d0a191,snowy,highway,daytime,daytime|snowy,test,val,18
82,b1e1a7b8-65ec7612.jpg,b1e1a7b8,rainy,highway,daytime,daytime|rainy,test,val,13
83,b1d22449-117aa773.jpg,b1d22449,clear,highway,night,night|clear,test,val,12


In [54]:
# 6. Data report

In [ ]:
def make_data_report(images_manifest, boxes_manifest, params: dict) -> str:
    """Markdown data report: size, class balance, conditions, small boxes, occlusion."""
    classes = params["classes"]
    img_tbl, box_tbl = images_manifest, boxes_manifest
    lines = ["# BDD100K subset: data report\n"]
 
    n_imgs = img_tbl.groupby("split").size().rename("images")
    n_boxes = box_tbl.groupby("split").size().rename("boxes")
    lines += ["## Size per split", "```", pd.concat([n_imgs, n_boxes], axis=1).to_string(), "```\n"]
 
    cls_counts = box_tbl.pivot_table(index="category", columns="split", values="name",
                                     aggfunc="count", fill_value=0).reindex(classes)
    cls_pct = (cls_counts / cls_counts.sum() * 100).round(1)
    lines += ["## Class counts (boxes)", "```", cls_counts.to_string(), "```",
              "## Class share per split (% of boxes)", "```", cls_pct.to_string(), "```\n"]
 
    for col in ["timeofday", "weather", "scene"]:
        t = pd.crosstab(img_tbl[col], img_tbl["split"], normalize="columns").mul(100).round(1)
        counts = pd.crosstab(img_tbl[col], img_tbl["split"])
        lines += [f"## {col}: % of images per split", "```", t.to_string(), "```",
                  f"## {col}: image counts per split", "```", counts.to_string(), "```\n"]
 
    small = (box_tbl.area < 32 * 32).groupby([box_tbl.split, box_tbl.category]).mean()
    small = small.mul(100).round(1).unstack(0).reindex(classes)
    occ = box_tbl.groupby("category")[["occluded", "truncated"]].mean().mul(100).round(1).reindex(classes)
    lines += ["## Small boxes (<32x32 px), % per class and split", "```", small.to_string(), "```",
              "## Occluded / truncated boxes, % per class", "```", occ.to_string(), "```\n"]
 
    return "\n".join(lines)